In [ ]:
!pip cache purge
!pip install --no-cache-dir -q accelerate peft bitsandbytes transformers trl


In [ ]:
!pip install -q --no-cache-dir accelerate
!pip install -q --no-cache-dir peft
!pip install -q --no-cache-dir bitsandbytes
!pip install -q --no-cache-dir transformers
!pip install -q --no-cache-dir trl


In [ ]:
!pip install --no-cache-dir accelerate==1.4.0 peft==0.5.0 bitsandbytes==0.45.2 transformers==4.49.0 trl==0.5.0


In [ ]:
!pip install --upgrade pip setuptools
!pip install -q torch datasets transformers bitsandbytes accelerate peft trl


In [ ]:
!pip install --no-cache-dir transformers==4.27.0

In [ ]:
!pip install datasets
!pip install --no-cache-dir transformers==4.33.0

In [5]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

/home/user/anaconda3/envs/dummy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"

# The instruction dataset to use
dataset_name = "mlabonne/guanaco-llama2-1k"

# Fine-tuned model name
new_model = "Llama-2-7b-chat-finetune"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 0

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

In [ ]:
from datasets import load_dataset
import torch
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig
from transformers import AutoModelForCausalLM # Import AutoModelForCausalLM


# Load dataset
dataset = load_dataset(dataset_name, split="train")

# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit, # This parameter was introduced in transformers==4.33.0
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},  # Load directly onto GPU 0
    trust_remote_code=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    # fp16=fp16,
    # bf16=bf16,
    fp16=False,
    bf16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

In [ ]:
!pip install --upgrade transformers accelerate

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load dataset from Hugging Face
# Replace 'your_dataset_name' with the actual dataset name from Hugging Face
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
# Optional: If your dataset is large, you can take a subset
# dataset = dataset.select(range(10000))

# 2. Load LLaMA 3 model and tokenizer
# Note: You'll need access to LLaMA 3, which might require authentication
# model_name = "meta-llama/Llama-3-8b"  # Example, adjust based on available version
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Example, adjust based on available version
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU
model.to(device)

# 3. Preprocess dataset
def preprocess_function(examples):
    # Adjust this based on your dataset's structure
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names,
)

# Split dataset into train and validation
train_val_split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_val_split["train"]
val_dataset = train_val_split["test"]

# 4. Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank of the low-rank matrices
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Modules to apply LoRA to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 5. Set up training arguments
training_args = TrainingArguments(
    output_dir="./llama3_lora_finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,  # Adjust based on your GPU memory (24GB for RTX 4090)
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=True,  # Enable mixed precision for faster training on RTX 4090
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    push_to_hub=False,  # Set to True if you want to push to Hugging Face Hub
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

# 7. Start training
trainer.train()

# 8. Save the fine-tuned model
model.save_pretrained("./llama3_lora_finetuned")
tokenizer.save_pretrained("./llama3_lora_finetuned")

# Optional: Push to Hugging Face Hub
# trainer.push_to_hub("your-username/llama3-lora-finetuned")

/home/user/anaconda3/envs/dummy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Map: 100%|██████████| 1000/1000 [00:00<00:00, 11742.67 examples/s]
/home/user/anaconda3/envs/dummy/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_22570/522541988.py:81: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


ValueError: The model did not return a loss from the inputs, only the following keys: logits,past_key_values. For reference, the inputs it received are input_ids,attention_mask.

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load dataset from Hugging Face
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

# 2. Load TinyLlama model and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU
model.to(device)

# 3. Preprocess dataset with labels
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names,
)

# Split dataset into train and validation
train_val_split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_val_split["train"]
val_dataset = train_val_split["test"]

# 4. Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 5. Set up training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama_lora_finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    # fp16=True,
    bf16=True,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",  # Explicitly define the metric for best model
    greater_is_better=False,  # Lower eval_loss is better
)

# 6. Initialize Trainer with a custom data collator (optional but recommended for PEFT)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

# 7. Start training
trainer.train()

# 8. Save the fine-tuned model (save both base model and adapter)
model.save_pretrained("./tinyllama_lora_finetuned")
tokenizer.save_pretrained("./tinyllama_lora_finetuned")

Using device: cuda


/home/user/anaconda3/envs/dummy/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_26008/895581081.py:85: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


Step,Training Loss,Validation Loss
100,1.049400,1.085693


('./tinyllama_lora_finetuned/tokenizer_config.json',
 './tinyllama_lora_finetuned/special_tokens_map.json',
 './tinyllama_lora_finetuned/tokenizer.json')

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from torch.utils.data import DataLoader

# 1. Load the fine-tuned model and tokenizer
model_path = "./tinyllama_lora_finetuned"
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

# Load base model and apply fine-tuned LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, model_path)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# 2. Define evaluation prompts
eval_prompts = [
    "Human: What is the capital of France? ### Assistant: ",
    "Human: Write a short poem about the moon. ### Assistant: ",
    "Human: How do I make a cup of tea? ### Assistant: ",
    "Human: Explain quantum physics in simple terms. ### Assistant: ",
]

# 3. Generate responses
print("Generating responses from the fine-tuned model...\n")
for prompt in eval_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response[len(prompt):]}\n")
    print('-'*70)

# 4.Quantitative evaluation: Compute perplexity on validation set
print("Computing perplexity on validation set...\n")

# Load and preprocess validation dataset
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
train_val_split = dataset.train_test_split(test_size=0.1)
val_dataset = train_val_split["test"]

def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

# Convert to PyTorch format
val_dataset = tokenized_val_dataset.with_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Use DataLoader for batching
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Compute perplexity
def compute_perplexity(model, dataloader):
    total_loss = 0.0
    total_tokens = 0
    model.eval()
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)
            labels = batch["labels"].to(model.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            total_loss += loss.item() * input_ids.size(1) * input_ids.size(0)  # seq_len * batch_size
            total_tokens += input_ids.size(1) * input_ids.size(0)
    
    perplexity = torch.exp(torch.tensor(total_loss / total_tokens))
    return perplexity.item()

# Calculate and print perplexity
perplexity = compute_perplexity(model, val_dataloader)
print(f"Perplexity on validation set: {perplexity:.2f}")

# Optional: Compare with base model
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.float16).to(model.device)
base_perplexity = compute_perplexity(base_model, val_dataloader)
print(f"Base model perplexity: {base_perplexity:.2f}")

Generating responses from the fine-tuned model...

Prompt: Human: What is the capital of France? ### Assistant: 
Response: ​The capital of France is Paris. 

----------------------------------------------------------------------
Prompt: Human: Write a short poem about the moon. ### Assistant: 
Response: 

The moon, a celestial beauty,
A shimmering silhouette,
A reminder of the great unknown,
A wonder to behold.

A constant companion,
A shining light,
A source of inspiration,
A reminder to rise.

The moon, a symbol of the unknown,
A beacon of hope,
A reminder to dream,
A journey of wonder.

The moon, a force

----------------------------------------------------------------------
Prompt: Human: How do I make a cup of tea? ### Assistant: 
Response: 

To make a cup of tea, you will need: 

- A tea bag or loose tea leaves 
- Water 
- A pot for boiling water 
- A teapot or infuser 
- A cup 
- A tea strainer or spoon 

Once you have all the ingredients, follow these steps: 

1. Add the tea ba

Map: 100%|██████████| 100/100 [00:00<00:00, 6803.74 examples/s]


Perplexity on validation set: 2.54
Base model perplexity: 349.40
